# Retrosynthesis exercise

Using the USPTO-50k dataset, here this exercise addresses the core principle of template-based retrosynthesis:

Product → Reaction template → Reactants

The first part of this notebook deals with only predicting the first step here, the reaction templates, i.e. here simply the reaction class. This simple classification model would become a policy in a multi-step search algorithm.

To make this a bit less abstract, in the second part of the notebook, reactants are recovered from the predicted reaction classes. Note that this is a simple data-driven approach and does include 
- atom mapping
- chemistry rules (hard‑coded)
- graph rewriting.

Hence, it is a conceptual demonstration, far from the prowess of more complex models. However, early systems would have adopted a very similar approach, so there is some real-world relevance of this exercise.

### Part 1: Predicting reaction classes

1) Setup

In [1]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import DataStructs, rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from tqdm import tqdm


2) Load Dataset

In [2]:
df = pd.read_csv("uspto-50k.csv")
df = df[["prod_smiles", "rxn_smiles", "rxn_class"]]
df["reactants"] = df["rxn_smiles"].str.split(">").str[0]

print("Number of reactions:", len(df))
df.head()

Number of reactions: 49015


,prod_smiles,rxn_smiles,rxn_class,reactants
0,COC(=O)C(C)c1ccc(-c2ccc(-c3nc(C(N)=O)c(C)nc3C)...,CC1(C)OB([c:1]2[cH:2][cH:3][c:4]([CH:5]([C:6](...,3,CC1(C)OB([c:1]2[cH:2][cH:3][c:4]([CH:5]([C:6](...
1,Cc1cc(-c2ccc(F)cc2)c2c(c1)cc1n2CCNC1=O,Br[c:1]1[cH:2][c:3]([CH3:4])[cH:5][c:6]2[c:7]1...,3,Br[c:1]1[cH:2][c:3]([CH3:4])[cH:5][c:6]2[c:7]1...
2,C[C@]1(F)CC(F)(F)[C@@](C)(c2cc(N)ccc2F)NC1=S,CC(C)(C)OC(=O)[NH:1][c:2]1[cH:3][cH:4][c:5]([F...,6,CC(C)(C)OC(=O)[NH:1][c:2]1[cH:3][cH:4][c:5]([F...
3,CCOC(=O)Cc1cccc(Oc2ccc(-c3c(C)noc3C)cc2CN2C(=O...,Br[c:1]1[c:2]([CH3:3])[n:4][o:5][c:6]1[CH3:7]....,3,Br[c:1]1[c:2]([CH3:3])[n:4][o:5][c:6]1[CH3:7]....
4,Nc1cc(CN2CCOCC2)cc(C(F)(F)F)c1,O=[C:1]([c:2]1[cH:3][c:4]([NH2:5])[cH:6][c:7](...,7,O=[C:1]([c:2]1[cH:3][c:4]([NH2:5])[cH:6][c:7](...


3) Featurisation
In retrosynthesis, only the product is known at the prediction time, so the product needs to be encoded (here: MorganFP).

In [ ]:
# helper function aka fingerprint nut just on the porduct
def smiles_to_fp(smiles, n_bits=2048, radius=2):
    fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    fp = fpgen.GetFingerprint(mol)
    fp_arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)
    return fp_arr

In [4]:
# Build the dataset
X = []
y = []
row_indices = [] 

for idx, row in df.iterrows():
    fp = smiles_to_fp(row["prod_smiles"])
    if fp is None:
        continue
    X.append(fp)
    y.append(row["rxn_class"])
    row_indices.append(idx)

X = np.array(X)
y = np.array(y)
row_indices = np.array(row_indices)

print("Usable samples:", len(X))
print("Number of classes:", len(set(y)))

Usable samples: 49015
Number of classes: 10


In [5]:
# Optional subsampling if too high load:
MAX_SAMPLES = 10000  # 30k took about 10 mins of training on my PC, 10k about 2.5 mins

if MAX_SAMPLES:
    idx = np.random.choice(len(X), MAX_SAMPLES, replace=False)
    X = X[idx]
    y = y[idx]
    row_indices = row_indices[idx]


X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, row_indices,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 8000
Test size: 2000


4) Train retrosynthesis classifier
Multinomial logistic regression is fast, interpretable, probabilistic (top-k!). Other models could be used as well, e.g. RF.

In [6]:
model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
    solver="saga"
)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, n_jobs=-1, solver='saga')

5) Evaluate top-k accuracy 

In [7]:
def top_k_accuracy(y_true, probs, k):
    top_k = np.argsort(probs, axis=1)[:, -k:]
    return np.mean([
        y_true[i] in top_k[i]
        for i in range(len(y_true))
    ])

In [8]:
probs = model.predict_proba(X_test)

for k in [1, 3, 5]:
    acc = top_k_accuracy(y_test, probs, k)
    print(f"Top-{k} accuracy: {acc:.3f}")

Top-1 accuracy: 0.064
Top-3 accuracy: 0.325
Top-5 accuracy: 0.575


6) Inspect some results:

In [9]:
i = np.random.randint(len(X_test))

true_class = y_test[i]
predicted_classes = np.argsort(probs[i])[-5:][::-1]

print("True reaction class:", true_class)
print("Top-5 predicted classes:", predicted_classes.tolist())

True reaction class: 8
Top-5 predicted classes: [7, 8, 1, 0, 4]


7) Discussion

- Why is retrosynthesis naturally a *ranking* problem?
- Why is accuracy higher than forward prediction?
    we have much more data per class then in the forward prediction (80000 classes in 100000 samples or someting like that)
- What chemical information is missing?
    yield, conditions, feasability
- How would this integrate into a multi-step planner?

### Part 2: Approximating the reactants

Idea: For each reaction class...
- collect all reactant SMILES
- keep the most frequent ones
- later, retrieve from this list

Result: Fast and robust approximation

1) Build reactant libraries per class

In [10]:
from collections import defaultdict, Counter

class_to_reactants_freq = defaultdict(list)

for cls, idx in zip(y_train, idx_train):
    reactants = df.loc[idx, "reactants"]
    class_to_reactants_freq[int(cls)].append(reactants)

for cls in class_to_reactants_freq:
    counts = Counter(class_to_reactants_freq[cls])
    class_to_reactants_freq[cls] = counts.most_common()


Inspect reaction classes:

In [11]:
example_class = list(class_to_reactants_freq.keys())[0]

print("Reaction class:", example_class)
print("Most common reactants:")
for r, count in class_to_reactants_freq[example_class][:5]:
    print(f"- {r} ({count}×)")


Reaction class: 2
Most common reactants:
- O[C:1]([C@@H:2]([NH:3][CH3:4])[C:5]([CH3:6])([CH3:7])[c:8]1[cH:9][cH:10][cH:11][s:12]1)=[O:13].[CH3:14][CH2:15][O:16][C:17](=[O:18])/[C:19]([CH3:20])=[CH:21]/[C@H:22]([CH:23]([CH3:24])[CH3:25])[N:26]([CH3:27])[C:28](=[O:29])[C@@H:30]([NH2:31])[C:32]([CH3:33])([CH3:34])[CH3:35] (2×)
- Cl[S:1](=[O:2])(=[O:3])[c:4]1[cH:5][c:6]([Cl:7])[cH:8][cH:9][c:10]1[Cl:11].[CH3:12][c:13]1[n:14][o:15][c:16]([NH2:17])[c:18]1[Br:19] (2×)
- [CH3:1][CH2:2][O:3][C:4](=[O:5])[c:6]1[cH:7][n:8][c:9]([N:10]2[CH2:11][CH2:12][NH:13][CH2:14][CH2:15]2)[c:16]([Cl:17])[cH:18]1.[O:19]=[C:20]=[N:21][CH2:22][c:23]1[cH:24][cH:25][cH:26][cH:27][cH:28]1 (1×)
- Cl[C:1]([c:2]1[cH:3][cH:4][c:5]([O:6][CH3:7])[cH:8][cH:9]1)=[O:10].[O:11]=[C:12]1[CH2:13][CH2:14][CH2:15][NH:16]1 (1×)
- O[CH:1]([CH2:2][CH2:3][CH3:4])[CH3:5].[CH3:6][C:7](=[O:8])[O:9][c:10]1[cH:11][cH:12][c:13]([C:14](=[O:15])[OH:16])[c:17]([F:18])[cH:19]1 (1×)


2) Predict Reactants Given a Predicted Reaction Class

In [12]:
"""
    Return top-k most frequent reactant sets
    for a given reaction class.
    """

def predict_reactants(predicted_class, k=5):
    predicted_class = int(predicted_class)
    
    if predicted_class not in class_to_reactants_freq:
        return []  # or raise a warning
    
    return [
        r for r, _ in class_to_reactants_freq[predicted_class][:k]
    ]


3) Combine with the class prediction:

In [13]:
i = np.random.randint(len(X_test))

# True values
true_class = y_test[i]

# CORRECT: map index -> class label
predicted_class = model.classes_[np.argmax(probs[i])]

predicted_reactants = predict_reactants(predicted_class, k=5)

print("True reaction class:", true_class)
print("Predicted reaction class:", predicted_class)

print("\nTop-5 predicted reactant sets:")
for r in predicted_reactants:
    print("-", r)

True reaction class: 6
Predicted reaction class: 5

Top-5 predicted reactant sets:
- CC(C)(C)OC(=O)O[C:6]([O:5][C:2]([CH3:1])([CH3:3])[CH3:4])=[O:7].[CH3:8][CH:9]([CH3:10])[N:11]([C:12](=[O:13])[CH2:14][N:15]1[C:16](=[O:17])[CH:18]([CH2:19][c:20]2[cH:21][nH:22][c:23]3[cH:24][cH:25][cH:26][cH:27][c:28]23)[c:29]2[n:30][n:31][c:32](-[c:33]3[cH:34][cH:35][cH:36][cH:37][cH:38]3)[n:39]2-[c:40]2[cH:41][cH:42][cH:43][cH:44][c:45]21)[c:46]1[cH:47][cH:48][cH:49][cH:50][cH:51]1
- CC(C)(C)OC(=O)O[C:6]([O:5][C:2]([CH3:1])([CH3:3])[CH3:4])=[O:7].[O:8]=[N+:9]([O-:10])[c:11]1[cH:12][cH:13][c:14]([NH:15][CH2:16][CH2:17][n:18]2[n:19][cH:20][cH:21][c:22]2[NH:23][C:24]([c:25]2[cH:26][cH:27][cH:28][cH:29][cH:30]2)([c:31]2[cH:32][cH:33][cH:34][cH:35][cH:36]2)[c:37]2[cH:38][cH:39][cH:40][cH:41][cH:42]2)[n:43][cH:44]1
- CC(C)(C)OC(=O)O[C:6]([O:5][C:2]([CH3:1])([CH3:3])[CH3:4])=[O:7].[O:8]=[C:9]([OH:10])[CH2:11][CH:12]1[CH2:13][CH2:14][NH:15][CH2:16][CH2:17]1
- CC(C)(C)OC(=O)O[C:6]([O:5][C:2]([CH3:1])([CH3:3])

4) Evaluation: Were the true reactants recovered? 

Use top-k retrieval accuracy:

In [14]:
def reactant_top_k_accuracy(
    y_true_classes,
    predicted_classes,
    true_reactants,
    k=5
):
    hits = 0
    for cls, true_r in zip(predicted_classes, true_reactants):
        candidates = predict_reactants(cls, k=k)
        if true_r in candidates:
            hits += 1
    return hits / len(true_reactants)

In [15]:
# Predict reaction classes for test set
predicted_classes = np.argmax(probs, axis=1)

true_reactants_test = df.loc[idx_test, "reactants"].values

for k in [1, 3, 5]:
    acc = reactant_top_k_accuracy(
        y_test,
        predicted_classes,
        true_reactants_test,
        k=k
    )
    print(f"Reactant Top-{k} accuracy: {acc:.3f}")

Reactant Top-1 accuracy: 0.000
Reactant Top-3 accuracy: 0.000
Reactant Top-5 accuracy: 0.000


5) Discussion

- Why is reactant prediction harder than reaction-class prediction?
- Why does top‑k accuracy matter more than top‑1?
- Why don't we retrieve our true reactants? How could this be remedied?
